## 🛠️ Data Preprocessing

In [228]:
import json
import pandas as pd
import os
from sklearn.model_selection import StratifiedGroupKFold

In [229]:
# Loading local dataset configurations (data paths)
with open("../config.json") as json_file:
    _LOCAL_CONFIG = json.load(json_file)

OUTPUT_FOLDER_PATH = _LOCAL_CONFIG['output_folder_path']
CSV_PATH = _LOCAL_CONFIG['dataset_csv_path']
MSKCC_CSV_PATH = _LOCAL_CONFIG['mskcc_csv_path']
IMAGES_PATH = _LOCAL_CONFIG['dataset_images_path']
TYPE_CLASS = "triage" # diag or triage
TYPE_IMG = "DERMATOSCOPE" # DERMATOSCOPE, CLINICAL or BOTH
TARGET_COLUMN = "histoMacroCID"
SECOND_TARGET_COLUMN = "clinicalMacroCID"
TARGET_NUMBER_COLUMN = "diagnostic-number"
IMG_COLUMN = "img-id"
IMG_SRC_COLUMN = "img-src"

### Data Import and Cleaning

In [230]:
clin_ = ["img-id", "img-src", "clinicalDiagnostic", "patient-id", "lesion-id", "histoDiagnostic", "clinicalMacroCID", "histoMacroCID"]
clin_feats = ["age", "usePesticide", "gender", "familySkinCancerHistory", "familyCancerHistory", 
              "fitzpatrickSkinType", "macroBodyRegion", "hasItched", "hasGrown", "hasHurt", "hasChanged", 
              "hasBled", "hasElevation"]


data_csv = pd.read_csv(CSV_PATH, dtype={4: str, 88: str, 89: str})

valid_columns = [c for c in clin_ + clin_feats if c in data_csv.columns]
data_csv = data_csv[valid_columns]
data_csv = data_csv.fillna("EMPTY").replace(" ", "EMPTY").replace("  ", "EMPTY").replace("NÃO  ENCONTRADO", "EMPTY")
data_csv["age"] = pd.to_numeric(data_csv["age"], errors='coerce').fillna(0).astype(int)

### Filtering by lesions with both clinical and dermatoscopical images

In [231]:
ids_clinical = set(data_csv[data_csv[IMG_SRC_COLUMN] == 'CLINICAL']['lesion-id'])
ids_dermatoscope = set(data_csv[data_csv[IMG_SRC_COLUMN] == 'DERMATOSCOPE']['lesion-id'])
ids_with_both = ids_clinical.intersection(ids_dermatoscope)

data_csv = data_csv[data_csv['lesion-id'].isin(ids_with_both)]

### Constructing the image path

In [232]:
data_csv['img-id'] = data_csv['img-id'].apply(lambda x: os.path.join(IMAGES_PATH, x + '.png'))

### In case of triage, joining the PAD-UFES-26 and MSKCC datasets

In [233]:
if TYPE_CLASS == "triage":
    mskcc_df = pd.read_csv(os.path.join(OUTPUT_FOLDER_PATH, 'mskcc_normal_skin_cli_derm_metadata.csv'), dtype={20: str})
    data_csv = pd.concat([data_csv, mskcc_df], ignore_index=True)

### Filtering by image source

In [234]:
if TYPE_IMG != "BOTH":
    data_csv = data_csv[data_csv[IMG_SRC_COLUMN] == TYPE_IMG]

### Class Clustering

In [235]:
# Selecting only the targets that we want
if TYPE_CLASS == "triage":
    cluster_targets = {
        "C43": "P1",
        "D03": "P1",
        "D22": "P1",

        "C80": "P2",
        "C44": "P2",
        "D04": "P2",
        "L75": "P2",  
        "D23": "P2",    

        "L57": "P3",
        "L25": "P3",
        "L30": "P3",
        "L98": "P3",    

        "L78": "P4",
        "L82": "P4",

        "L70": "P5",
        "00": "P5"
    }
elif TYPE_CLASS == "diag":
    cluster_targets = {
        "C43": "MEL",
        "D03": "MEL",
        "D22": "NEVO",

        "C80": "CBC",
        "C44": "CEC",
        "D04": "CEC",

        "L57": "ACT",    

        "L78": "NEVO",
        "L82": "SEBO",
    }

for idx, row in data_csv.iterrows():
    if row[TARGET_COLUMN] in cluster_targets:
        data_csv.at[idx, TARGET_COLUMN] = cluster_targets[row[TARGET_COLUMN]]
    elif row[SECOND_TARGET_COLUMN] in cluster_targets:
        data_csv.at[idx, TARGET_COLUMN] = cluster_targets[row[SECOND_TARGET_COLUMN]]
    else:        
        data_csv.drop(idx, inplace=True)

print("- Checking the target distribution")
print(data_csv[TARGET_COLUMN].value_counts())
print(f"Total number of samples: {data_csv[TARGET_COLUMN].value_counts().sum()}")

- Checking the target distribution
histoMacroCID
P2    2112
P3    2102
P5    1810
P4     154
P1     124
Name: count, dtype: int64
Total number of samples: 6302


### Metadata Consolidation

In [236]:
# reset dataframe index
data_csv = data_csv.reset_index(drop=True)

# Selecting the clinical features
cli_feats = dict()
for idx, row in data_csv.iterrows():
    img_id = row[IMG_COLUMN]

    anamnese = ""
    for col in clin_feats:
        anamnese += f"{col}: {row[col]} "    
    
    cli_feats[img_id] = anamnese    

### K-Fold Generation

In [237]:
# Splitting the dataset
data_csv['lesion-id'] = data_csv['lesion-id'].astype(str)
data_train = data_csv[[IMG_COLUMN, TARGET_COLUMN, 'lesion-id']].copy()
kfold = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
data_train['folder'] = None
for i, (train_indexes, test_indexes) in enumerate(kfold.split(data_train, data_train[TARGET_COLUMN], groups=data_train['lesion-id'])):
    data_train.loc[test_indexes, 'folder'] = i + 1 

# Converting the labels to numbers
data_train[TARGET_COLUMN] = data_train[TARGET_COLUMN].astype('category')
data_train[TARGET_NUMBER_COLUMN] = data_train[TARGET_COLUMN].cat.codes

data_train.to_csv(os.path.join(OUTPUT_FOLDER_PATH, f"pad-ufes-26-{TYPE_CLASS}_{TYPE_IMG.lower()}_folders_raw.csv"), index=False)

# Saving the data raw
tokens_path = os.path.join(OUTPUT_FOLDER_PATH, f"anamnese_raw_{TYPE_CLASS}_{TYPE_IMG.lower()}.json")
with open(tokens_path, 'w') as f:
    json.dump(cli_feats, f, indent=4)
